# Practice Assignment 3 (Week 3): A reproducible air-data pipeline
# تمرین تمرینی ۳ (هفته‌ی ۳): خط لوله‌ی تکرارپذیر داده‌های هوایی

**Course:** AI for Aerospace & Mechanical Engineering (PhD), Block 0 &nbsp;|&nbsp;
**Released:** Week 3 &nbsp;|&nbsp; **Due:** start of Week 4 &nbsp;|&nbsp; **Effort:** about 3.5 hours

> **Provenance:** new content. Dataclass and ABC usage follows (rewritten, not copied)
> [W3Schools dataclasses Module](https://www.w3schools.com/python/ref_module_dataclasses.asp),
> [W3Schools abc Module](https://www.w3schools.com/Python/ref_module_abc.asp),
> [GeeksforGeeks Understanding Python Dataclasses](https://www.geeksforgeeks.org/python/understanding-python-dataclasses/),
> [GeeksforGeeks Abstract Classes in Python](https://www.geeksforgeeks.org/python/abstract-classes-in-python/) and
> [GeeksforGeeks Getting Started with Pytest](https://www.geeksforgeeks.org/python/getting-started-with-pytest/).
> Min-max scaling is adapted from Géron, *Hands-On Machine Learning*, 3rd ed., Ch. 2.

<div dir="rtl">

## صورت مسئله

در هفته‌ی ۶ مدلی خواهیم ساخت که مصرف سوخت را از پارامترهای پرواز پیش‌بینی می‌کند. ورودی آن مدل از همان لاگ ۵۰ هرتزی پرواز هفته‌ی ۱ می‌آید: ارتفاع بارومتری، سرعت هوا و شتاب عمودی. این لاگ سه ایراد دارد: سه جهش تک‌نمونه‌ای در `accel_z_mps2` (حدود ۵ g که برای یک IMU با بازه‌ی ±۴g ناممکن است)، ۷۵ نمونه‌ی گم‌شده (NaN) در ارتفاع و سرعت هوا، و یک وقفه‌ی ۱٫۲ ثانیه‌ای در ثبت داده در حدود ثانیه‌ی ۱۰۰. در آزمایشگاه دیدید که میانگین متحرکی که «سطرها» را می‌شمارد و نه «ثانیه‌ها»، در کنار این وقفه یک نرخ صعود غیرواقعی (حدود ۲۱ متر بر ثانیه) می‌سازد.

وظیفه‌ی شما ساختن یک خط لوله‌ی **تکرارپذیر** است که داده‌ی آماده‌ی مدل تولید کند:

۱. کلاس `PlausibleRange`: مقادیر ناممکن فیزیکی را به NaN تبدیل کند (نه اینکه آن‌ها را به حد بازه ببُرد).
۲. کلاس `Resample`: همه‌ی ستون‌ها را روی یک شبکه‌ی زمانی یکنواخت (۱۰ هرتز) درون‌یابی کند تا NaNها و وقفه پر شوند.
۳. کلاس `MinMaxScale`: مرحله‌ای که از داده **یاد می‌گیرد** (کمینه و بیشینه) و آن را فقط روی داده‌ی آموزشی برازش می‌دهیم.
۴. تابع‌های `build_airdata_pipeline`، `split_fit_transform`، `save_pipeline` و `load_pipeline`: هفت مرحله را به ترتیب مشخص به هم وصل کنید، روی ۷۰٪ ابتدایی پرواز برازش دهید و پیکربندی را به‌صورت JSON ذخیره و بازسازی کنید.
۵. دست‌کم ۶ آزمون pytest برای کلاس‌های خودتان بنویسید.

هر سه کلاس زیرکلاس `PipelineStage` از کتابخانه‌ی درس (`src/aerospace_ai/pipeline.py`) و از نوع dataclass هستند. کد را در فایل `airdata_stages.py` و آزمون‌ها را در `test_airdata_stages.py` بنویسید؛ این دفترچه فقط ماژول شما را بارگذاری و بررسی می‌کند. همه‌ی توضیحات داخل کد (comment) باید انگلیسی باشند.

</div>

**English summary.** Write three dataclass stages on top of the course `PipelineStage` ABC
(`PlausibleRange`, `Resample`, `MinMaxScale`), assemble the seven-stage air-data pipeline
specified below, fit it on the first 70 % of the flight only, save and reload its configuration
as JSON, and write your own pytest suite. Code goes in `airdata_stages.py`, tests in
`test_airdata_stages.py`.

**The pipeline (Task 4), in this order:**

| # | Stage | Settings |
|---|-------|----------|
| 1 | `SelectColumns` | `time_s`, `baro_alt_m`, `airspeed_mps`, `accel_z_mps2` |
| 2 | `PlausibleRange` | `accel_z_mps2` in [−4 g₀, +4 g₀] |
| 3 | `Resample` | 10 Hz on `time_s` |
| 4 | `HampelFilter` | `baro_alt_m`, `airspeed_mps`; window 7 |
| 5 | `RollingMean` | all three signals; window 5 |
| 6 | `TimeDerivative` | `baro_alt_m` → `climb_mps` |
| 7 | `MinMaxScale` | all three signals and `climb_mps`; range (0, 1) |

Use the module constants (`IMU_RANGE_G`, `RESAMPLE_RATE_HZ`, ...) instead of typing numbers.

## Setup (run first, do not edit)

In [ ]:
from __future__ import annotations

import importlib
import json
import math
import subprocess
import sys
from pathlib import Path
from types import ModuleType
from typing import Callable, Final

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    """Walk up from ``start`` until the folder that contains ``src/aerospace_ai``."""
    for folder in (start, *start.parents):
        if (folder / "src" / "aerospace_ai").is_dir():
            return folder
    raise FileNotFoundError("Open this notebook from inside the course repository.")


REPO_ROOT: Final[Path] = find_repo_root(Path.cwd())
ASSIGNMENT_DIR: Final[Path] = REPO_ROOT / "assignments" / "practice03"
for folder in (REPO_ROOT / "src", ASSIGNMENT_DIR):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

FLIGHT_LOG: Final[Path] = REPO_ROOT / "data" / "sample" / "week01_uav_flight_log.csv"
CONFIG_PATH: Final[Path] = ASSIGNMENT_DIR / "airdata_pipeline.json"
G0: Final[float] = 9.80665


def load_student_module() -> ModuleType:
    """Import (or re-import after an edit) the student's airdata_stages.py."""
    module = importlib.import_module("airdata_stages")
    return importlib.reload(module)


def status(task: str, func: Callable[[], None]) -> None:
    """Run a public check; print [ ] while the task still raises NotImplementedError."""
    try:
        func()
    except NotImplementedError:
        print(f"[ ] {task}: not done yet")
        return
    print(f"[x] {task}: public checks passed")


ap = load_student_module()
log = pd.read_csv(FLIGHT_LOG)
print("flight log:", log.shape, "| NaN per column:", log.isna().sum()[lambda s: s > 0].to_dict())

### Background: what you inherit

`PipelineStage` (abstract, generic) gives every stage `__call__` (input-type check, fitted check,
output-type check), `fit` (no-op by default), `is_fitted`, `name`, `get_params` (the dataclass
fields with `init=True`) and `>>`. Your stages declare `INPUT_TYPE`/`OUTPUT_TYPE` (already
written for you) and implement `transform`. `DataPipeline` checks the chain, runs it, reports,
and round-trips through `to_config`/`from_config`. Read `src/aerospace_ai/pipeline.py`,
especially `HampelFilter` (a frozen stage) and `Standardize` (a stage that learns), before you
start.

### Task 1 (15 pts): `PlausibleRange`
<div dir="rtl">یک مرحله‌ی dataclass منجمد (frozen) بنویسید که ستون‌ها را به tuple تبدیل کند، مرزهای نامعتبر را رد کند و مقادیر خارج از بازه‌ی بسته‌ی [lower, upper] را NaN کند. داده‌ی ورودی نباید تغییر کند.</div>

Frozen dataclass; in `__post_init__` store `columns` as a tuple (a single string means one
column) with `object.__setattr__`, and raise `ValueError` if a bound is not finite or
`lower >= upper`. `transform` returns a **copy** in which values outside the **closed**
interval become NaN. On the flight log with ±4 g₀ on `accel_z_mps2` exactly 3 values become NaN.

In [ ]:
def _check_task1() -> None:
    ap = load_student_module()
    stage = ap.PlausibleRange(["accel_z_mps2"], -4 * G0, 4 * G0)
    print(stage)
    out = stage(log)
    n_nan = int(out["accel_z_mps2"].isna().sum())
    print("accel_z values turned into NaN:", n_nan)
    assert n_nan == 3 and log["accel_z_mps2"].notna().all()


status("Task 1", _check_task1)

In [ ]:
# HIDDEN TESTS (task1): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 2 (20 pts): `Resample`
<div dir="rtl">همه‌ی ستون‌های عددی را روی شبکه‌ی زمانی یکنواخت از نخستین تا آخرین زمان درون‌یابی خطی کنید. هر ستون فقط از نقاط غیر NaN خودش استفاده می‌کند. ستون‌های غیرعددی حذف می‌شوند.</div>

Frozen dataclass with `rate_hz` and `time_column` (default `"time_s"`). Reject a rate that
is not positive and finite. `period_s` is a read-only property. `grid(t_start, t_end)` returns
$t_\text{start} + k/f$ for $k = 0, 1, \dots$ while the point does not exceed $t_\text{end}$
(round-off: `grid(0.0, 0.3)` at 10 Hz has **4** points). `transform` interpolates each numeric
column through its own non-NaN samples with `np.interp`; a column that is all NaN stays all NaN.

*By hand:* time $[0, 0.3, 1.0]$ s, $x = [0, 3, 10]$, 2 Hz. Grid $[0, 0.5, 1.0]$; at 0.5 s,
between $(0.3, 3)$ and $(1.0, 10)$: $x = 3 + 7 \times 0.2/0.7 = 5.0$.

In [ ]:
def _check_task2() -> None:
    ap = load_student_module()
    small = pd.DataFrame({"time_s": [0.0, 0.3, 1.0], "x": [0.0, 3.0, 10.0]})
    out = ap.Resample(2.0)(small)
    print(out)
    assert out["time_s"].tolist() == [0.0, 0.5, 1.0] and math.isclose(out["x"].iloc[1], 5.0)
    resampled = ap.Resample(10.0)(log[["time_s", "baro_alt_m"]])
    print("flight log at 10 Hz:", resampled.shape, "| NaN left:", int(resampled.isna().sum().sum()))
    assert resampled.shape == (3000, 2)


status("Task 2", _check_task2)

In [ ]:
# HIDDEN TESTS (task2): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 3 (25 pts): `MinMaxScale`, a stage that learns
<div dir="rtl">این مرحله کمینه و بیشینه‌ی هر ستون را در <code>fit</code> یاد می‌گیرد. فیلدهای یادگرفته‌شده نباید آرگومان سازنده باشند، نباید چاپ شوند و نباید در مقایسه شرکت کنند. پیش از برازش، <code>transform</code> باید <code>NotFittedError</code> بدهد.</div>

A **non-frozen** dataclass (`fit` changes it) with `columns` and `feature_range=(0.0, 1.0)`.
Learned fields `data_min_`, `data_max_`: `dict[str, float] | None`, declared with
`field(default=None, init=False, repr=False, compare=False)`. `fit` ignores NaN, rejects a
constant column (`ValueError`) and returns `self`. With $(a, b)$ = `feature_range`:

$$x' = a + (x - x_\min)\,\frac{b-a}{x_\max - x_\min},\qquad
x = x_\min + (x' - a)\,\frac{x_\max - x_\min}{b-a}\ \ (\texttt{inverse\_transform}).$$

*By hand:* train $\{2, 4, 6\}$, range $(-1, 1)$: $x=4 \mapsto -1 + 2\cdot 2/4 = 0$;
$x = 10 \mapsto -1 + 8\cdot 2/4 = 3$ (test data may leave the range; that is information, not an
error).

In [ ]:
def _check_task3() -> None:
    ap = load_student_module()
    scaler = ap.MinMaxScale(["x"], feature_range=(-1.0, 1.0))
    print(scaler, "| fitted:", scaler.is_fitted)
    scaler.fit(pd.DataFrame({"x": [2.0, 4.0, 6.0]}))
    out = scaler(pd.DataFrame({"x": [4.0, 10.0]}))
    print("scaled:", out["x"].tolist(), "| learned:", scaler.data_min_, scaler.data_max_)
    assert np.allclose(out["x"], [0.0, 3.0]) and scaler == ap.MinMaxScale(["x"], (-1.0, 1.0))


status("Task 3", _check_task3)

In [ ]:
# HIDDEN TESTS (task3): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 4 (25 pts): assemble, fit without leakage, save, reload
<div dir="rtl">هفت مرحله را به ترتیب جدول بالا وصل کنید. داده را بر اساس زمان تقسیم کنید: ۷۰٪ ابتدایی سطرها آموزش و بقیه آزمون. فقط روی بخش آموزش برازش دهید و هر دو بخش را تبدیل کنید. پیکربندی را به JSON با تورفتگی ذخیره کنید و از آن دوباره خط لوله را بسازید.</div>

* `build_airdata_pipeline()` returns the unfitted seven-stage pipeline of the table above.
* `split_fit_transform(pipeline, log, train_fraction=0.7)`: the first
  `int(len(log) * train_fraction)` rows are the training part (a **time** split, never a random
  one for a time series); raise `ValueError` unless $0 <$ `train_fraction` $< 1$; fit on the
  training part only; return `(train_output, test_output)`.
* `save_pipeline(pipeline, path)` writes `pipeline.to_config()` as JSON with `indent=2` and
  returns the path; `load_pipeline(path)` rebuilds an unfitted pipeline with
  `DataPipeline.from_config`.

In [ ]:
def _check_task4() -> None:
    ap = load_student_module()
    pipeline = ap.build_airdata_pipeline()
    print(repr(pipeline))
    train_out, test_out = ap.split_fit_transform(pipeline, log)
    print("train", train_out.shape, "| test", test_out.shape)
    print(test_out.describe().loc[["min", "max"]].round(3))
    path = ap.save_pipeline(pipeline, CONFIG_PATH)
    print("saved to", Path(path).name, "| reloaded == original:", ap.load_pipeline(path) == pipeline)
    report = pipeline.run(log).report_frame()
    print(report[["stage", "rows_in", "rows_out", "nan_out", "cells_changed"]])
    assert pipeline.is_fitted and train_out.shape == (2104, 5)


status("Task 4", _check_task4)

In [ ]:
# HIDDEN TESTS (task4): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

### Task 5 (15 pts): your pytest suite
<div dir="rtl">در فایل <code>test_airdata_stages.py</code> دست‌کم ۶ تابع آزمون بنویسید (دو نمونه داده شده است) و از <code>pytest.raises</code>، <code>parametrize</code> و یک fixture استفاده کنید. مدرس آزمون‌های شما را روی سه پیاده‌سازی عمداً معیوب نیز اجرا می‌کند و برای هر خطایی که آزمون‌هایتان پیدا کنند امتیاز می‌گیرید.</div>

At least 6 test functions (the two examples count) using `pytest.raises`,
`@pytest.mark.parametrize` and a fixture; small hand-made frames only. Scoring: 3 points if
there are at least 6 tests and all pass on the reference solution, 3 for using the three
features, and 3 for each of three hidden broken implementations your tests catch.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "-p", "no:warnings",
     "test_airdata_stages.py"],
    cwd=ASSIGNMENT_DIR, capture_output=True, text=True,
)
print(result.stdout[-2000:])

In [ ]:
# HIDDEN TESTS (task5): the instructor's grader replaces this cell.
# Leave it in place and do not edit it.

## Visual check (not graded)

Climb rate from the barometer near the 1.2 s logger gap (t ≈ 100 s): the row-counting pipeline
of the lab (Exercise 8.1) against your resampled pipeline (stages 1 to 6). Your curve should not
jump at the gap.

In [ ]:
try:
    ap = load_student_module()
    from aerospace_ai.pipeline import FillGaps, HampelFilter, RollingMean, TimeDerivative

    row_based = (
        FillGaps(["baro_alt_m"], max_gap=5, time_column="time_s")
        >> HampelFilter(["baro_alt_m"], window=7)
        >> RollingMean(["baro_alt_m"], window=51)
        >> TimeDerivative("baro_alt_m", "climb_mps", time_column="time_s")
    )(log)
    resampled = ap.build_airdata_pipeline()[:6](log)
    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(row_based["time_s"], row_based["climb_mps"], lw=0.8, label="lab Exercise 8.1 (rows)")
    ax.plot(resampled["time_s"], resampled["climb_mps"], lw=1.0, label="your pipeline (10 Hz grid)")
    ax.axvspan(99.98, 101.2, color="0.85", label="logger gap")
    ax.set(xlim=(90, 112), xlabel="time [s]", ylabel="climb rate [m/s]")
    ax.legend(fontsize=8)
    plt.show()
except NotImplementedError:
    print("[ ] finish Tasks 1 to 4 first")

## Expected output (check yourself against these)

| Quantity | Expected value |
|----------|----------------|
| Flight log shape / NaN in `baro_alt_m`, `airspeed_mps` | (14940, 10) / 75, 75 |
| `accel_z_mps2` values outside ±4 g₀ | 3 |
| `Resample(2.0)` on the hand example | time [0, 0.5, 1.0], x[1] = 5.0 |
| Flight log resampled to 10 Hz | 3000 rows, no NaN |
| `MinMaxScale((-1, 1))` fitted on {2, 4, 6}, applied to {4, 10} | 0.0, 3.0 |
| Max climb rate after stages 1 to 6 | 19.91 m/s at t = 84.0 s (lab, row-based: 20.99 m/s at 99.76 s) |
| Train / test output shapes | (2104, 5) / (897, 5) |
| Learned `data_max_["baro_alt_m"]` | 600.974 m |
| Test-set minimum of scaled `climb_mps` | −0.768 (the final descent is steeper than anything in training) |
| Stage report: cells changed by `PlausibleRange` / `HampelFilter` | 3 / 231 |
| Reloaded pipeline == original | True (and not fitted) |

## Rubric (100 points)

| Task | Points | Full marks when | Typical deductions |
|------|-------:|-----------------|--------------------|
| 1. `PlausibleRange` | 15 | frozen, tuple columns, bounds validated, closed interval, NaN not clip, input untouched | clipping (−8); `<` instead of `<=` (−3) |
| 2. `Resample` | 20 | validated rate, `period_s` property, round-off-safe grid, NaN bridged, non-numeric dropped | `np.arange(t0, t1, dt)` length errors (−5) |
| 3. `MinMaxScale` | 25 | learned fields `init/repr/compare=False`, `NotFittedError`, exact formulas both ways, constant column rejected | learned state compared (−5); ignores `feature_range` (−8) |
| 4. Pipeline, split, save/load | 25 | exact stage order and settings, time split, fit on train only, indented JSON round trip | fitting on all rows (−10) |
| 5. Your pytest suite | 15 | ≥ 6 tests passing on the reference (3), `raises` + `parametrize` + fixture (3), 3 per hidden broken implementation caught (9) | tests that only check types catch nothing |

Tasks 1 to 5 are auto-graded by hidden instructor tests. **Code-quality gate** (up to −10): type
hints on every signature, Google-style docstrings, English-only comments, module constants
instead of magic numbers, Black formatting (88 characters), and *Restart & Run All* completes
without errors.